Configuración y Limpieza

In [6]:
import pandas as pd
import numpy as np

# 1. Carga de datos
df_calendar = pd.read_csv('../data/calendar_limpio.csv.zip', compression='zip', parse_dates=['date'])
df_listings = pd.read_csv('../data/listings_limpio.csv')
df_rentas = pd.read_csv('../data/renta_sevilla_capital_limpio.csv')

# Merge: Cruce para tener el barrio en cada fecha
airbnb = pd.merge(df_calendar, df_listings[['id', 'neighbourhood_cleansed', 'room_type', 'price']], 
             left_on='listing_id', right_on='id', how='left')

# 2. Ingeniería de Fechas (Efecto Fiestas en Sevilla)
def etiquetar_evento(fecha):
    # Ejemplo Feria de Abril 2024 (14-20 abril)
    if '2024-04-14' <= str(fecha.date()) <= '2024-04-20':
        return 'Feria de Abril'
    # Ejemplo Semana Santa 2024 (24-31 marzo)
    elif '2024-03-24' <= str(fecha.date()) <= '2024-03-31':
        return 'Semana Santa'
    else:
        return 'Normal'

# Crear nuevas columnas basadas en la fecha y el evento.
airbnb['evento'] = airbnb['date'].apply(etiquetar_evento)
airbnb['mes'] = airbnb['date'].dt.month_name()
airbnb['dia_semana'] = airbnb['date'].dt.day_name()
airbnb['es_finde'] = airbnb['date'].dt.dayofweek.isin([4, 5, 6]) # Viernes, Sábado, Domingo

# Temporada Alta/Baja (Criterio de negocio)
# Definimos Alta como los meses de primavera/verano en Sevilla (abril a junio y septiembre a octubre)
airbnb['temporada'] = airbnb['date'].dt.month.apply(
    lambda x: 'Alta' if x in [3, 4, 5, 9, 10] else 'Baja'
)

airbnb.columns

Index(['listing_id', 'date', 'available', 'minimum_nights', 'maximum_nights',
       'id', 'neighbourhood_cleansed', 'room_type', 'price', 'evento', 'mes',
       'dia_semana', 'es_finde', 'temporada'],
      dtype='object')

Inflación de los precios por las fiestas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Agrupamos por temporada para sacar el precio medio
# Usamos 'price_x' que es el precio diario del calendario
resumen_precios = df.groupby('temporada')['price_x'].mean()

# 2. Cálculo de la Inflación (%)
precio_normal = resumen_precios['Normal']
precio_feria = resumen_precios['Feria de Abril']
precio_ssanta = resumen_precios['Semana Santa']

inflacion_feria = ((precio_feria - precio_normal) / precio_normal) * 100
inflacion_ssanta = ((precio_ssanta - precio_normal) / precio_normal) * 100

print(f"💰 Precio Medio Normal: {precio_normal:.2f}€")
print(f"🎡 Precio Medio Feria: {precio_feria:.2f}€ ({inflacion_feria:.1f}% de subida)")
print(f"⛪ Precio Medio Semana Santa: {precio_ssanta:.2f}€ ({inflacion_ssanta:.1f}% de subida)")

# 3. Visualización: Comparativa Visual
plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

# Creamos el gráfico de barras
colores = ['#95a5a6', '#e74c3c', '#f1c40f'] # Gris para normal, Rojo para Feria, Amarillo para SS
grafico = sns.barplot(x=resumen_precios.index, y=resumen_precios.values, palette=colores)

# Añadimos etiquetas de datos encima de las barras para que sea más legible
for p in grafico.patches:
    grafico.annotate(format(p.get_height(), '.2f') + '€', 
                     (p.get_x() + p.get_width() / 2., p.get_height()), 
                     ha = 'center', va = 'center', 
                     xytext = (0, 9), 
                     textcoords = 'offset points',
                     fontweight='bold')

plt.title('El "Efecto Fiestas" en Sevilla: Impacto en Precios de Airbnb', fontsize=15)
plt.ylabel('Precio Medio por Noche (€)')
plt.xlabel('Periodo')
plt.ylim(0, max(resumen_precios.values) * 1.2) # Damos espacio arriba para las etiquetas

plt.show()